In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import tensorflow as tf
from tensorflow.keras import layers

In [2]:
cols = ["Survived", "Pclass", "Sex", "Age", "Fare", "Embarked", "Title", "FamilySize", "HasCabin"]

data = pd.read_csv("../data/Titanic-Dataset2.csv")
data = data[cols]
data

,Survived,Pclass,Sex,Age,Fare,Embarked,Title,FamilySize,HasCabin
0,0,3,male,22.000000,7.2500,S,Mr,1,0
1,1,1,female,38.000000,71.2833,C,Mrs,1,1
2,1,3,female,26.000000,7.9250,S,Miss,0,0
3,1,1,female,35.000000,53.1000,S,Mrs,1,1
4,0,3,male,35.000000,8.0500,S,Mr,0,0
...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.000000,13.0000,S,Other,0,0
887,1,1,female,19.000000,30.0000,S,Miss,0,1
888,0,3,female,21.845638,23.4500,S,Miss,3,0
889,1,1,male,26.000000,30.0000,C,Mr,0,1


In [3]:
categorical_cols = ["Survived", "Pclass", "Sex", "Embarked", "Title", "HasCabin"]
continuous_cols = ["Age", "Fare", "FamilySize"]

In [4]:
data[categorical_cols]

,Survived,Pclass,Sex,Embarked,Title,HasCabin
0,0,3,male,S,Mr,0
1,1,1,female,C,Mrs,1
2,1,3,female,S,Miss,0
3,1,1,female,S,Mrs,1
4,0,3,male,S,Mr,0
...,...,...,...,...,...,...
886,0,2,male,S,Other,0
887,1,1,female,S,Miss,1
888,0,3,female,S,Miss,0
889,1,1,male,C,Mr,1


In [5]:
data[continuous_cols]

,Age,Fare,FamilySize
0,22.000000,7.2500,1
1,38.000000,71.2833,1
2,26.000000,7.9250,0
3,35.000000,53.1000,1
4,35.000000,8.0500,0
...,...,...,...
886,27.000000,13.0000,0
887,19.000000,30.0000,0
888,21.845638,23.4500,3
889,26.000000,30.0000,0


In [6]:
ohe = OneHotEncoder(sparse_output=False)
scaler = StandardScaler()

X_cat = ohe.fit_transform(data[categorical_cols])
X_cont = scaler.fit_transform(data[continuous_cols])

X = np.concatenate([X_cont, X_cat], axis=1)
X = X.astype("float32")

data_dim = X.shape[1]

In [7]:
ohe.feature_names_in_

array(['Survived', 'Pclass', 'Sex', 'Embarked', 'Title', 'HasCabin'],
      dtype=object)

In [8]:
ohe.categories_

[array([0, 1]),
 array([1, 2, 3]),
 array(['female', 'male'], dtype=object),
 array(['C', 'Q', 'S'], dtype=object),
 array(['Master', 'Miss', 'Mr', 'Mrs', 'Other'], dtype=object),
 array([0, 1])]

In [9]:
X.shape

(891, 20)

In [10]:
cat_dims = [len(ohe.categories_[i]) for i in range(len(categorical_cols))]
cond_dim = sum(cat_dims)

def sample_condition(batch_size):
    cond = np.zeros((batch_size, cond_dim))
    for i, dim in enumerate(cat_dims):
        idx = np.random.randint(0, dim, batch_size)
        start = sum(cat_dims[:i])
        cond[np.arange(batch_size), start + idx] = 1
    return tf.convert_to_tensor(cond, dtype=tf.float32)


In [11]:
print(cat_dims, cond_dim)

[2, 3, 2, 3, 5, 2] 17


In [12]:
sample_condition(5)

2026-02-12 19:47:41.374279: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


<tf.Tensor: shape=(5, 17), dtype=float32, numpy=
array([[1., 0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 1., 0., 0., 0., 0.,
        1.],
       [0., 1., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0.,
        1.],
       [1., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0., 1.,
        0.],
       [1., 0., 1., 0., 0., 1., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0.,
        1.],
       [0., 1., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0.,
        1.]], dtype=float32)>

In [13]:
latent_dim = 128

def build_generator():
    z = layers.Input(shape=(latent_dim,))
    cond = layers.Input(shape=(cond_dim,))
    x = layers.Concatenate()([z, cond])

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(256, activation="relu")(x)
    out = layers.Dense(data_dim)(x)

    return tf.keras.Model([z, cond], out)


In [14]:
def build_discriminator():
    x_in = layers.Input(shape=(data_dim,))
    cond = layers.Input(shape=(cond_dim,))
    x = layers.Concatenate()([x_in, cond])

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(256, activation="relu")(x)
    out = layers.Dense(1)(x)

    return tf.keras.Model([x_in, cond], out)


In [15]:
def gradient_penalty(discriminator, real, fake, cond):
    alpha = tf.random.uniform([real.shape[0], 1], 0., 1.)
    interpolated = alpha * real + (1 - alpha) * fake

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        pred = discriminator([interpolated, cond], training=True)

    grads = tape.gradient(pred, interpolated)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1))
    return tf.reduce_mean((norm - 1.0) ** 2)


In [16]:
generator = build_generator()
discriminator = build_discriminator()

g_opt = tf.keras.optimizers.Adam(1e-5, beta_1=1e-5)
d_opt = tf.keras.optimizers.Adam(1e-5, beta_1=1e-5)

batch_size = 256
epochs = 300
lambda_gp = 10
critic_steps = 5

dataset = tf.data.Dataset.from_tensor_slices(X).shuffle(2000).batch(batch_size)

for epoch in range(epochs):
    for real_batch in dataset:
        for _ in range(critic_steps):
            z = tf.random.normal([real_batch.shape[0], latent_dim])
            cond = sample_condition(real_batch.shape[0])

            with tf.GradientTape() as tape:
                fake = generator([z, cond], training=True)
                d_real = discriminator([real_batch, cond], training=True)
                d_fake = discriminator([fake, cond], training=True)

                gp = gradient_penalty(discriminator, real_batch, fake, cond)
                d_loss = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real) + lambda_gp * gp

            grads = tape.gradient(d_loss, discriminator.trainable_variables)
            d_opt.apply_gradients(zip(grads, discriminator.trainable_variables))

        z = tf.random.normal([real_batch.shape[0], latent_dim])
        cond = sample_condition(real_batch.shape[0])

        with tf.GradientTape() as tape:
            fake = generator([z, cond], training=True)
            g_loss = -tf.reduce_mean(discriminator([fake, cond], training=True))

        grads = tape.gradient(g_loss, generator.trainable_variables)
        g_opt.apply_gradients(zip(grads, generator.trainable_variables))

    if epoch % 50 == 0:
        print(f"Epoch {epoch} | D: {d_loss.numpy():.3f} | G: {g_loss.numpy():.3f}")


2026-02-12 19:47:42.702984: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 0 | D: 5.275 | G: -0.154


2026-02-12 19:47:43.714087: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-12 19:47:45.722690: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-12 19:47:49.776802: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-12 19:47:57.832424: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-12 19:48:13.922185: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 50 | D: 0.440 | G: -1.358


2026-02-12 19:48:46.166042: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 100 | D: -0.913 | G: 0.171


2026-02-12 19:49:51.052004: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 150 | D: -0.474 | G: -0.096
Epoch 200 | D: -0.658 | G: -0.661
Epoch 250 | D: -0.905 | G: -0.238


2026-02-12 19:52:01.333544: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [17]:
z = tf.random.normal([1000, latent_dim])
cond = sample_condition(1000)

synthetic = generator([z, cond], training=False).numpy()

X_cont_syn = scaler.inverse_transform(synthetic[:, :len(continuous_cols)])
X_cat_syn = ohe.inverse_transform(synthetic[:, len(continuous_cols):])

synthetic_df = pd.DataFrame(
    np.column_stack([X_cont_syn, X_cat_syn]),
    columns=continuous_cols + categorical_cols
)

print(synthetic_df.head(50))


          Age       Fare FamilySize Survived Pclass     Sex Embarked   Title  \
0   18.579533   7.663938  -0.122628        1      1    male        S    Miss   
1   26.524746 -28.670105   0.316105        0      1    male        S      Mr   
2   39.848743  22.534521  -0.614376        1      1  female        S  Master   
3    6.618529 -18.685966   2.796183        0      3  female        C    Miss   
4   27.290607  37.681316  -0.001682        0      2    male        S      Mr   
5   43.721672  12.474169   0.152842        1      3    male        S      Mr   
6   22.819691  11.697815   2.308646        0      1  female        S    Miss   
7   34.482628   23.74622   1.460075        0      3    male        S   Other   
8   37.653507  80.919441   1.966719        0      3    male        S  Master   
9   29.462872  12.688423  -0.139052        0      3    male        S    Miss   
10   13.94585 -26.926994   1.273211        0      3    male        C    Miss   
11  29.865425  20.682451   0.631639     

In [18]:
#print(synthetic_df[synthetic_df["profesion"] == "desempleado"])